In [2]:
import pandas as pd
import numpy as np
import geopandas as gpd
import re
import unicodedata

RUTA = ""   # vacío porque el notebook está en la misma carpeta que los archivos

barrios = gpd.read_file(RUTA + r"C:\Users\jhonn\Desktop\NEOLAND\PROYECTO\PROYECTO_BOOTCAMP_ALQUILER\01_datos_originales\Barrios_131_oficial.topojson")

print("Forma:", barrios.shape)
print()
print(barrios.dtypes)
barrios.head()

# El archivo viene en latitud/longitud (WGS84), pero no lo indica: se lo decimos
barrios = barrios.set_crs(4326, allow_override=True)

Forma: (131, 13)

id              object
CODDIS          object
NOMDIS          object
COD_BAR         object
NOMBRE          object
COD_DIS_TX      object
BARRIO_MAY      object
COD_DISBAR      object
NUM_BAR         object
BARRIO_MT       object
COD_DISB        object
Area            object
geometry      geometry
dtype: object


In [3]:
# --- Fase 2: calidad del dato ---
print("Nulos por columna:")
print(barrios[["COD_DIS_TX", "NOMDIS", "COD_BAR", "NOMBRE"]].isnull().sum())
print()
print("Barrios repetidos (por nombre):", barrios["NOMBRE"].duplicated().sum())
print("Número de distritos:", barrios["NOMDIS"].nunique())
print("Polígonos mal dibujados:", (~barrios.is_valid).sum())

# --- Limpieza ---
# 1) Reparar polígonos con pequeños defectos de dibujo (no cambia los límites)
barrios["geometry"] = barrios.buffer(0)

# 2) Crear el número oficial: distrito (2 cifras) + barrio (3 cifras)
#    zfill rellena con ceros a la izquierda: "1" -> "01", "11" -> "011"
barrios["cod_distrito"] = barrios["COD_DIS_TX"].str.zfill(2)
barrios["id_barrio"] = barrios["cod_distrito"] + barrios["COD_BAR"].str.zfill(3)

# 3) Quedarnos solo con lo útil y con nombres claros
barrios = barrios[["id_barrio", "cod_distrito", "NOMDIS", "NOMBRE", "geometry"]]
barrios = barrios.rename(columns={"NOMDIS": "nombre_distrito", "NOMBRE": "nombre_barrio"})

# --- Comprobaciones ---
print()
print("Forma final:", barrios.shape)
print("¿id_barrio único?:", barrios["id_barrio"].is_unique)
barrios.head()

Nulos por columna:
COD_DIS_TX    0
NOMDIS        0
COD_BAR       0
NOMBRE        0
dtype: int64

Barrios repetidos (por nombre): 0
Número de distritos: 21
Polígonos mal dibujados: 18

Forma final: (131, 5)
¿id_barrio único?: True


,id_barrio,cod_distrito,nombre_distrito,nombre_barrio,geometry
0,17171,17,Villaverde,Villaverde Alto - Casco Histórico de Villaverde,"POLYGON ((-3.69297 40.34585, -3.69295 40.34584..."
1,17172,17,Villaverde,San Cristóbal,"POLYGON ((-3.69255 40.33274, -3.69259 40.33408..."
2,17173,17,Villaverde,Butarque,"POLYGON ((-3.65752 40.32893, -3.65989 40.32789..."
3,17174,17,Villaverde,Los Rosales,"POLYGON ((-3.68379 40.35024, -3.68379 40.35022..."
4,17175,17,Villaverde,Ángeles,"POLYGON ((-3.69401 40.36453, -3.69363 40.36173..."


In [4]:
# 1) Pasar el mapa a metros para poder medir distancias
barrios_metros = barrios.to_crs(25830)

# 2) Un punto dentro de cada barrio
#    representative_point() siempre cae DENTRO del barrio
#    (el centroide puede caer fuera si el barrio tiene forma rara)
punto_barrio = barrios_metros.representative_point()

# 3) La Puerta del Sol, también en metros
sol = gpd.GeoSeries.from_xy([-3.7035285], [40.4169473], crs=4326).to_crs(25830).iloc[0]

# 4) Distancia de cada barrio a Sol, en km
barrios["distancia_centro_km"] = (punto_barrio.distance(sol) / 1000).round(3)

# 5) Guardar la latitud y longitud de ese punto (para los mapas y el modelo espacial)
punto_latlon = punto_barrio.to_crs(4326)
barrios["lat_c"] = punto_latlon.y.round(6)
barrios["lon_c"] = punto_latlon.x.round(6)

# --- Comprobaciones ---
print("Superficie total (km²):", round(barrios_metros.area.sum() / 1e6, 1))
print(barrios["distancia_centro_km"].describe().round(2))
print()
print("5 barrios más céntricos:")
print(barrios.nsmallest(5, "distancia_centro_km")[["nombre_barrio", "distancia_centro_km"]])

Superficie total (km²): 603.9
count    131.00
mean       5.68
std        3.15
min        0.08
25%        3.26
50%        5.21
75%        7.36
max       17.19
Name: distancia_centro_km, dtype: float64

5 barrios más céntricos:
   nombre_barrio  distancia_centro_km
60           Sol                0.078
59        Cortes                0.508
52   Embajadores                0.838
67   Universidad                0.972
64      Justicia                1.011


In [5]:
# Cargar el precio del alquiler por zona (Idealista, agosto 2026)
idealista = pd.read_excel(RUTA + r"C:\Users\jhonn\Desktop\NEOLAND\PROYECTO\PROYECTO_BOOTCAMP_ALQUILER\01_datos_originales\Idealista_Precios.xlsx")

# --- Fase 1: conocer el dataset ---
print("Forma:", idealista.shape)
print()
print(idealista.dtypes)
print()

# --- Fase 2: calidad del dato ---
print("Nulos por columna:")
print(idealista.isnull().sum())
print()
print("Zonas repetidas:", idealista["zona_idealista"].duplicated().sum())
print("Número de distritos:", idealista["distrito"].nunique())

# Valores imposibles: un alquiler en Madrid no baja de 5 ni pasa de 60 €/m²
fuera = idealista[(idealista["precio_m2_ago2026"] < 5) | (idealista["precio_m2_ago2026"] > 60)]
print("Precios imposibles:", len(fuera))
print()
print(idealista["precio_m2_ago2026"].describe().round(2))
idealista.head()

Forma: (104, 3)

distrito              object
zona_idealista        object
precio_m2_ago2026    float64
dtype: object

Nulos por columna:
distrito             0
zona_idealista       0
precio_m2_ago2026    0
dtype: int64

Zonas repetidas: 0
Número de distritos: 21
Precios imposibles: 0

count    104.00
mean      21.10
std        3.54
min       15.10
25%       18.73
50%       20.40
75%       23.32
max       31.60
Name: precio_m2_ago2026, dtype: float64


,distrito,zona_idealista,precio_m2_ago2026
0,Centro,Chueca-Justicia,27.9
1,Centro,Huertas-Cortes,26.1
2,Centro,Lavapiés-Embajadores,25.7
3,Centro,Malasaña-Universidad,25.9
4,Centro,Palacio,24.6


In [7]:
# Función para comparar nombres escritos de forma distinta
# Ej.: "Fuencarral - El Pardo" y "Fuencarral-El Pardo" -> "fuencarral el pardo"
def normalizar(texto):
    texto = unicodedata.normalize("NFKD", str(texto)).encode("ascii", "ignore").decode().lower()
    texto = re.sub(r"[^a-z0-9 ]", " ", texto)
    return re.sub(r"\s+", " ", texto).strip()

# Zonas de Idealista cuyo nombre NO coincide con el barrio oficial.
# Las de barrios nuevos (Las Tablas, Montecarmelo, Sanchinarro...) se verificaron
# cruzando anuncios con coordenadas contra los polígonos oficiales.
ZONA_A_BARRIO = {
    "Chueca-Justicia": "Justicia",
    "Huertas-Cortes": "Cortes",
    "Lavapiés-Embajadores": "Embajadores",
    "Malasaña-Universidad": "Universidad",
    "Buena Vista": "Buenavista",
    "Pau de Carabanchel": "Buenavista",
    "Bernabéu-Hispanoamérica": "Hispanoamérica",
    "Nuevos Ministerios-Ríos Rosas": "Ríos Rosas",
    "Tres Olivos - Valverde": "Valverde",
    "Las Tablas": "Valverde",
    "Montecarmelo": "El Goloso",
    "Conde Orgaz-Piovera": "Piovera",
    "Sanchinarro": "Valdefuentes",
    "Valdebebas - Valdefuentes": "Valdefuentes",
    "Virgen del Cortijo - Manoteras": "Valdefuentes",
    "Cuzco-Castillejos": "Castillejos",
    "Ventilla-Almenara": "Almenara",
    "Ensanche de Vallecas - La Gavia": "Ensanche de Vallecas",
    "Villaverde Alto": "Villaverde Alto - Casco Histórico de Villaverde",
    "Jerónimos": "Los Jerónimos",
    "Concepción": "La Concepción",
    "Salvador": "El Salvador",
}

# Diccionario de la tabla oficial: nombre normalizado -> id_barrio
nombre_a_id = dict(zip(barrios["nombre_barrio"].map(normalizar), barrios["id_barrio"]))

# 1) Traducir la zona a su nombre oficial (si no está en el diccionario, se queda igual)
idealista["barrio_oficial"] = idealista["zona_idealista"].replace(ZONA_A_BARRIO)

# 2) Buscar su número
idealista["id_barrio"] = idealista["barrio_oficial"].map(normalizar).map(nombre_a_id)

# --- Comprobación 1: ninguna zona sin número ---
sin_numero = idealista[idealista["id_barrio"].isna()]
print("Zonas sin id_barrio:", len(sin_numero))
print(sin_numero[["distrito", "zona_idealista"]])

# --- Comprobación 2: cada zona cae en SU distrito ---
# (las 2 primeras cifras del id_barrio son el código de distrito)
distrito_a_cod = dict(zip(barrios["nombre_distrito"].map(normalizar), barrios["cod_distrito"]))
idealista["cod_distrito"] = idealista["distrito"].map(normalizar).map(distrito_a_cod)
mal_distrito = idealista[idealista["cod_distrito"] != idealista["id_barrio"].str[:2]]
print("Zonas asignadas a otro distrito:", len(mal_distrito))

# --- Varias zonas en el mismo barrio -> media ---
zonas_por_barrio = idealista.groupby("id_barrio")["zona_idealista"].count()
print()
print("Barrios con más de una zona:")
print(zonas_por_barrio[zonas_por_barrio > 1])

precio = idealista.groupby("id_barrio")["precio_m2_ago2026"].mean().round(2).reset_index()
precio = precio.rename(columns={"precio_m2_ago2026": "precio_m2"})
print()
print("Barrios con precio:", precio.shape[0])
precio.head()

Zonas sin id_barrio: 0
Empty DataFrame
Columns: [distrito, zona_idealista]
Index: []
Zonas asignadas a otro distrito: 0

Barrios con más de una zona:
id_barrio
08086    2
11116    2
16166    3
Name: zona_idealista, dtype: int64

Barrios con precio: 100


,id_barrio,precio_m2
0,01011,24.6
1,01012,25.7
2,01013,26.1
3,01014,27.9
4,01015,25.9


In [9]:
# Cargar Airbnb (si tienes el CSV original, usa la segunda línea y borra la primera)
airbnb = pd.read_csv(RUTA + r"C:\Users\jhonn\Desktop\NEOLAND\PROYECTO\PROYECTO_BOOTCAMP_ALQUILER\01_datos_originales\Airbnb_Alquileres.csv")
# airbnb = pd.read_csv(RUTA + "airbnb_listings_2026_06.csv")

# --- Fase 1: conocer el dataset ---
print("Forma:", airbnb.shape)
print()
print(airbnb[["id", "latitude", "longitude", "room_type", "number_of_reviews_ltm"]].dtypes)
print()

# --- Fase 2: calidad del dato ---
print("Nulos en columnas clave:")
print(airbnb[["id", "latitude", "longitude", "room_type", "number_of_reviews_ltm"]].isnull().sum())
print()
print("Anuncios repetidos (id):", airbnb["id"].duplicated().sum())

# --- CONTROL ANTI-CORRUPCIÓN: Madrid está entre latitud 40,3-40,7 y longitud -3,9 y -3,5 ---
coords_malas = airbnb[~airbnb["latitude"].between(40.3, 40.7) | ~airbnb["longitude"].between(-3.9, -3.5)]
print("Coordenadas fuera de Madrid (posible corrupción):", len(coords_malas))
print()
print(airbnb[["latitude", "longitude"]].describe().round(4))
print()
print(airbnb["room_type"].value_counts())

Forma: (22835, 19)

id                         int64
latitude                 float64
longitude                float64
room_type                 object
number_of_reviews_ltm      int64
dtype: object

Nulos en columnas clave:
id                       0
latitude                 0
longitude                0
room_type                0
number_of_reviews_ltm    0
dtype: int64

Anuncios repetidos (id): 0
Coordenadas fuera de Madrid (posible corrupción): 0

         latitude   longitude
count  22835.0000  22835.0000
mean      40.4226     -3.6937
std        0.0236      0.0277
min       40.3319     -3.8321
25%       40.4101     -3.7073
50%       40.4208     -3.7008
75%       40.4330     -3.6850
max       40.5355     -3.5459

room_type
Entire home/apt    15899
Private room        6738
Shared room          159
Hotel room            39
Name: count, dtype: int64


In [10]:
# 1) y 2) Viviendas completas y activas
airbnb_ca = airbnb[(airbnb["room_type"] == "Entire home/apt") &
                   (airbnb["number_of_reviews_ltm"] > 0)].copy()
print("Anuncios totales:", len(airbnb))
print("Viviendas completas activas:", len(airbnb_ca))

# 3) Convertir cada anuncio en un punto del mapa
puntos = gpd.GeoDataFrame(
    airbnb_ca,
    geometry=gpd.points_from_xy(airbnb_ca["longitude"], airbnb_ca["latitude"]),
    crs=4326,
)

# ¿En qué barrio cae cada punto? (how="left" para ver también los que no caen en ninguno)
puntos = gpd.sjoin(puntos, barrios[["id_barrio", "geometry"]], how="left", predicate="within")

# Un punto justo en la línea entre dos barrios puede casar con ambos: nos quedamos con uno
puntos = puntos.drop_duplicates(subset="id")

print("Anuncios fuera del municipio (sin barrio):", puntos["id_barrio"].isna().sum())
puntos = puntos.dropna(subset=["id_barrio"])
print("Anuncios asignados a un barrio:", len(puntos))

# 4) Contar anuncios por barrio
n_airbnb = puntos.groupby("id_barrio")["id"].count().reset_index()
n_airbnb = n_airbnb.rename(columns={"id": "n_airbnb"})

print()
print("Barrios con al menos 1 Airbnb:", len(n_airbnb), "de 131")
print()
print("Top 5 barrios con más Airbnb:")
print(n_airbnb.merge(barrios[["id_barrio", "nombre_barrio"]], on="id_barrio")
              .nlargest(5, "n_airbnb"))

Anuncios totales: 22835
Viviendas completas activas: 12027
Anuncios fuera del municipio (sin barrio): 1
Anuncios asignados a un barrio: 12026

Barrios con al menos 1 Airbnb: 127 de 131

Top 5 barrios con más Airbnb:
  id_barrio  n_airbnb nombre_barrio
1     01012      1463   Embajadores
4     01015      1142   Universidad
0     01011      1060       Palacio
5     01016       773           Sol
2     01013       664        Cortes


In [11]:
# Cargar sin cabecera: el Excel tiene títulos y notas mezclados
renta_bruta = pd.read_excel(RUTA + r"C:\Users\jhonn\Desktop\NEOLAND\PROYECTO\PROYECTO_BOOTCAMP_ALQUILER\01_datos_originales\Renta_Media_Madrid_Barrio.xlsx", header=None)

# --- Fase 1: ver cómo viene ---
print("Forma del Excel original:", renta_bruta.shape)
print(renta_bruta.head(10))
print()

# Poner nombre a las 4 columnas
renta_bruta.columns = ["distrito", "territorio", "renta_media_persona", "renta_media_hogar"]
renta_bruta["territorio"] = renta_bruta["territorio"].astype(str).str.strip()

# Quedarse SOLO con las filas de barrio (empiezan por 3 cifras + punto: "011. Palacio")
renta = renta_bruta[renta_bruta["territorio"].str.match(r"^\d{3}\.\s")].copy()

# Crear el id_barrio: 2 cifras del distrito ("01. Centro" -> "01") + 3 del barrio ("011")
renta["id_barrio"] = renta["distrito"].astype(str).str[:2] + renta["territorio"].str[:3]

# Asegurar que la renta es numérica
renta["renta_media_persona"] = pd.to_numeric(renta["renta_media_persona"], errors="coerce")
renta = renta[["id_barrio", "territorio", "renta_media_persona"]]

# --- Fase 2: calidad del dato ---
print("Forma final:", renta.shape)
print("Nulos en la renta:", renta["renta_media_persona"].isnull().sum())
print("id_barrio repetidos:", renta["id_barrio"].duplicated().sum())

# ¿Todos los números existen en la tabla oficial de barrios?
no_cruzan = set(renta["id_barrio"]) - set(barrios["id_barrio"])
print("id_barrio que no existen en la tabla oficial:", no_cruzan if no_cruzan else "ninguno")
print()
print(renta["renta_media_persona"].describe().round(2))
renta.head()

Forma del Excel original: (158, 4)
                  0                 1                             2  \
0               NaN               NaN  Renta neta media por persona   
1               NaN               NaN                          2023   
2  Ciudad de Madrid  Ciudad de Madrid                        19.632   
3        01. Centro        01. Centro                        22.073   
4        01. Centro      011. Palacio                        23.770   
5        01. Centro  012. Embajadores                        18.177   
6        01. Centro       013. Cortes                        24.671   
7        01. Centro     014. Justicia                        28.051   
8        01. Centro  015. Universidad                        22.689   
9        01. Centro          016. Sol                        21.020   

                            3  
0  Renta neta media por hogar  
1                        2023  
2                      49.916  
3                      44.958  
4                      

,id_barrio,territorio,renta_media_persona
4,01011,011. Palacio,23.770
5,01012,012. Embajadores,18.177
6,01013,013. Cortes,24.671
7,01014,014. Justicia,28.051
8,01015,015. Universidad,22.689


In [12]:
excel = pd.ExcelFile(RUTA + r"C:\Users\jhonn\Desktop\NEOLAND\PROYECTO\PROYECTO_BOOTCAMP_ALQUILER\01_datos_originales\Indicadores_distritos_barrios_2024.xlsx")

# Hojas de distrito: las que empiezan por 2 cifras y un punto ("01. Centro", ...)
hojas = [h for h in excel.sheet_names if re.match(r"^\d{2}\.", h)]
print("Hojas de distrito:", len(hojas))

# Indicadores que queremos: nombre EXACTO en el Excel -> nombre corto
INDICADORES = {
    "Número Habitantes": "poblacion",
    "Número de inmuebles de uso residencial": "n_viviendas",
    "Tasa de crecimiento demográfico (porcentaje)": "crec_demografico_pct",
}

filas = []
for hoja in hojas:
    tabla = pd.read_excel(excel, sheet_name=hoja, header=None)
    etiquetas = tabla.iloc[:, 1].astype(str).str.strip()   # columna con los nombres de indicadores
    cabecera = tabla.iloc[1]                               # fila con "011. Palacio", "012. Embajadores"...

    for col, texto in cabecera.items():
        if re.match(r"^\d{3}\.\s", str(texto)):            # es una columna de barrio
            fila = {"id_barrio": hoja[:2] + str(texto)[:3]}
            for nombre_excel, nombre_corto in INDICADORES.items():
                n_fila = tabla.index[etiquetas == nombre_excel][0]
                fila[nombre_corto] = tabla.iloc[n_fila, col + 1]   # el valor está en la columna de al lado
            filas.append(fila)

panel = pd.DataFrame(filas)
panel[["poblacion", "n_viviendas", "crec_demografico_pct"]] = \
    panel[["poblacion", "n_viviendas", "crec_demografico_pct"]].apply(pd.to_numeric, errors="coerce")

# --- Antes de corregir: el crecimiento viene con el signo invertido ---
print("Barrios con crecimiento NEGATIVO (antes de corregir):", (panel["crec_demografico_pct"] < 0).sum())

# Corrección: el Ayuntamiento publica salidas - entradas. Sale negativo en TODA la ciudad,
# incluso en barrios en plena expansión como El Cañaveral (-27 %). Se invierte el signo.
panel["crec_demografico_pct"] = -panel["crec_demografico_pct"]

# --- Fase 2: calidad del dato ---
print()
print("Forma:", panel.shape)
print("Nulos:")
print(panel.isnull().sum())
print("id_barrio repetidos:", panel["id_barrio"].duplicated().sum())
no_cruzan = set(panel["id_barrio"]) - set(barrios["id_barrio"])
print("id_barrio que no existen en la tabla oficial:", no_cruzan if no_cruzan else "ninguno")
print()
print("Población total de Madrid:", int(panel["poblacion"].sum()))
print("Barrios con crecimiento positivo (después de corregir):", (panel["crec_demografico_pct"] > 0).sum())
print()
print(panel.describe().round(2))

Hojas de distrito: 21
Barrios con crecimiento NEGATIVO (antes de corregir): 131

Forma: (131, 4)
Nulos:
id_barrio               0
poblacion               0
n_viviendas             0
crec_demografico_pct    0
dtype: int64
id_barrio repetidos: 0
id_barrio que no existen en la tabla oficial: ninguno

Población total de Madrid: 3527924
Barrios con crecimiento positivo (después de corregir): 131

       poblacion  n_viviendas  crec_demografico_pct
count     131.00       131.00                131.00
mean    26930.72     11627.77                  3.46
std     15234.82      6459.07                  2.62
min      1696.00       580.00                  0.28
25%     17716.50      7010.50                  2.28
50%     23939.00     10791.00                  3.08
75%     36234.50     14738.50                  4.39
max     75829.00     30415.00                 27.37


In [13]:
# 1) Partir de la lista oficial (sin la columna del polígono)
df = barrios[["id_barrio", "cod_distrito", "nombre_distrito", "nombre_barrio",
              "lat_c", "lon_c", "distancia_centro_km"]].copy()

# 2) Pegar cada fuente por id_barrio
df = df.merge(precio,   on="id_barrio", how="left", validate="one_to_one")
df = df.merge(n_airbnb, on="id_barrio", how="left", validate="one_to_one")
df = df.merge(renta[["id_barrio", "renta_media_persona"]], on="id_barrio", how="left", validate="one_to_one")
df = df.merge(panel,    on="id_barrio", how="left", validate="one_to_one")

# 3) Barrio sin anuncios = 0 Airbnb (no es un dato que falte)
df["n_airbnb"] = df["n_airbnb"].fillna(0)

# 4) Variable de la hipótesis: viviendas Airbnb por cada 100 viviendas del barrio
df["densidad_airbnb"] = (100 * df["n_airbnb"] / df["n_viviendas"]).round(4)

# 5) Quedarse con las variables del proyecto
columnas = ["id_barrio", "cod_distrito", "nombre_distrito", "nombre_barrio", "lat_c", "lon_c",
            "precio_m2",                                         # Y
            "densidad_airbnb",                                   # X de la hipótesis
            "renta_media_persona", "distancia_centro_km", "crec_demografico_pct"]   # controles
df = df[columnas].sort_values("id_barrio").reset_index(drop=True)

# --- Comprobaciones del dataset completo ---
print("Dataset completo:", df.shape)
print("Nulos por columna:")
print(df.isnull().sum())
print("Barrios con densidad 0:", df.loc[df["densidad_airbnb"] == 0, "nombre_barrio"].tolist())

# 6) Dataset del modelo: solo barrios con precio
df_modelo = df.dropna(subset=["precio_m2"]).reset_index(drop=True)
print()
print("Dataset del modelo:", df_modelo.shape)
print("Nulos:", df_modelo.isnull().sum().sum())
print("Barrios con densidad 0:", (df_modelo["densidad_airbnb"] == 0).sum())
print()
print(df_modelo.describe().round(3))

# 7) Guardar (Excel para abrirlo sin que se estropee; CSV para Python)
df.to_excel(RUTA + "dataset_barrios_completo.xlsx", index=False)
df_modelo.to_excel(RUTA + "dataset_barrios_modelo.xlsx", index=False)
df_modelo.to_csv(RUTA + "dataset_barrios_modelo.csv", index=False)
print("\nGuardado ✅")

Dataset completo: (131, 11)
Nulos por columna:
id_barrio                0
cod_distrito             0
nombre_distrito          0
nombre_barrio            0
lat_c                    0
lon_c                    0
precio_m2               31
densidad_airbnb          0
renta_media_persona      0
distancia_centro_km      0
crec_demografico_pct     0
dtype: int64
Barrios con densidad 0: ['El Pardo', 'Horcajo', 'Atalaya', 'Valdebernardo']

Dataset del modelo: (100, 11)
Nulos: 0
Barrios con densidad 0: 0

         lat_c    lon_c  precio_m2  densidad_airbnb  renta_media_persona  \
count  100.000  100.000    100.000          100.000              100.000   
mean    40.427   -3.682     21.188            0.995               22.024   
std      0.037    0.045      3.573            2.185                7.097   
min     40.340   -3.783     15.100            0.022                9.910   
25%     40.401   -3.708     18.875            0.191               15.338   
50%     40.427   -3.688     20.450          